# TinyStories LM — Kaggle T4 training

Boilerplate to run `src/train.py` on Kaggle's free T4 GPU. It clones the repo, installs deps, points the code at the tokenized dataset via env vars, and starts training.

**Before running, in the right-hand panel:**
1. **Settings → Accelerator → GPU T4 x1**
2. **Settings → Internet → On** (needed to clone GitHub + log to Weights & Biases)
3. **Add Input →** add your private dataset that holds `train.bin`, `val.bin`, `tokenizer.json` (see Task 6.2 / `scripts/kaggle_dataset/`).
4. **Add-ons → Secrets →** add `WANDB_API_KEY` (optional; without it, wandb logs offline).

Then **Run All**. Checkpoints are written to `my-LLM/checkpoints/` under `/kaggle/working`, which is saved as the notebook's output.

## 1. GPU sanity check

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — set Accelerator to 'GPU T4 x1' in the right panel.")

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/rishipadhye/my-LLM.git"
REPO_DIR = "/kaggle/working/my-LLM"

if os.path.isdir(REPO_DIR):
    print("repo already present — pulling latest")
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

## 3. Install dependencies
`torch`, `numpy`, `tqdm`, and `PyYAML` come preinstalled on Kaggle; only `wandb` and `tokenizers` need installing for training.

In [ ]:
!pip install -q wandb tokenizers

## 4. Point the code at the dataset (env vars, no symlinks)
The dataset mounts read-only under `/kaggle/input/<slug>/`. `src/paths.py` reads `DATA_DIR` and `TOKENIZER_PATH` from the environment (falling back to the local `data/` and `tokenizer/` layout), so we just export them here — the `!python` training cell below inherits this kernel's environment.

In [ ]:
import os, glob

def find_one(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"{name} not found under /kaggle/input — did you add the dataset via 'Add Input'?")
    return hits[0]

# train.bin and val.bin live in the same dataset dir; point DATA_DIR there.
os.environ["DATA_DIR"] = os.path.dirname(find_one("train.bin"))
os.environ["TOKENIZER_PATH"] = find_one("tokenizer.json")

print("DATA_DIR        =", os.environ["DATA_DIR"])
print("TOKENIZER_PATH  =", os.environ["TOKENIZER_PATH"])

## 5. Weights & Biases auth
Loads `WANDB_API_KEY` from Kaggle Secrets if present; otherwise falls back to offline logging so training still runs.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("WANDB_API_KEY loaded from Kaggle Secrets — logging online")
except Exception as e:
    os.environ["WANDB_MODE"] = "offline"
    print("No WANDB_API_KEY secret — running wandb in offline mode:", e)

## 6. Train
Run from the repo root so `configs/` and `checkpoints/` resolve. Override `CONFIG_PATH` / `CKPT_DIR` here too if you want a different config or checkpoint dir.

In [ ]:
%cd /kaggle/working/my-LLM
!python src/train.py